# Hamiltonians with Braket

Build Pauli Hamiltonians as matrices and compute expectation values on quantum circuits simulated with `LocalSimulator`.

In [ ]:
import numpy as np
from braket.circuits import Circuit, ResultType
from braket.devices import LocalSimulator

## Pauli matrices and Hamiltonian construction

In [ ]:
I2 = np.eye(2, dtype=complex)
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

def pauli_string(ops):
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

H2_PAULIS = [
    (-0.81261, [I2, I2]),
    (0.17120, [PAULI_Z, I2]),
    (-0.22279, [I2, PAULI_Z]),
    (0.17120, [PAULI_Z, PAULI_Z]),
    (0.04532, [PAULI_X, PAULI_X]),
]
H2_MATRIX = sum(coeff * pauli_string(ops) for coeff, ops in H2_PAULIS)

eigenvalues = np.linalg.eigvalsh(H2_MATRIX)
print("Pauli decomposition:")
labels = ["II", "ZI", "IZ", "ZZ", "XX"]
for (coeff, _), label in zip(H2_PAULIS, labels):
    print(f"  {coeff:+.5f} · {label}")
print(f"\nEigenvalues: {np.round(eigenvalues, 6)}")
print(f"Ground state energy: {eigenvalues[0]:.6f}")

## Expectation values via simulation

In [ ]:
def statevector_from_circuit(circuit):
    n_qubits = max(circuit.qubit_count, 2)
    for q in range(n_qubits):
        circuit.i(q)
    circuit.add_result_type(ResultType.StateVector())
    device = LocalSimulator()
    task = device.run(circuit, shots=0)
    return np.array(task.result().result_types[0].value, dtype=complex)

def basis_state(idx, n_qubits=2):
    psi = np.zeros(2**n_qubits, dtype=complex)
    psi[idx] = 1.0
    return psi

def expval_basis(idx, hamiltonian):
    psi = basis_state(idx)
    return float(np.real(psi.conj() @ hamiltonian @ psi))

print(f"<00|H|00> = {expval_basis(0, H2_MATRIX):.6f}")
print(f"<11|H|11> = {expval_basis(3, H2_MATRIX):.6f}")

# |++⟩ via circuit
c = Circuit()
c.h(0)
c.h(1)
sv = statevector_from_circuit(c)
e_pp = float(np.real(sv.conj() @ H2_MATRIX @ sv))
print(f"<++|H|++> = {e_pp:.6f}")

## ZZ + transverse field lattice Hamiltonian

In [ ]:
J, H_FIELD = 1.0, 0.5
LATTICE_MATRIX = (
    J * pauli_string([PAULI_Z, PAULI_Z])
    + H_FIELD * pauli_string([PAULI_X, I2])
    + H_FIELD * pauli_string([I2, PAULI_X])
)
evals = np.linalg.eigvalsh(LATTICE_MATRIX)
print(f"H = {J:.1f}·Z₀Z₁ + {H_FIELD:.1f}·(X₀ + X₁)")
print(f"Eigenvalues: {np.round(evals, 6)}")
print(f"Ground state energy: {evals[0]:.6f}")
print(f"<01|H|01> = {expval_basis(1, LATTICE_MATRIX):.6f}")